# LaLonde (1986) Case Study ATT

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
# URLs for Dehejia and Wahba's LaLonde dataset components
treatment_url = "https://users.nber.org/~rdehejia/data/nswre74_treated.txt"
control_url = "http://www.nber.org/~rdehejia/data/cps_controls.txt" # Provided by user

# Column names for the Dehejia-Wahba version of the LaLonde dataset
# These are standard for the raw treated and control files.
column_names_dw = [
    'treat', 'age', 'education', 'black', 'hispan',
    'married', 'nodegree', 're74', 're75', 're78'
]

# Load the treated dataset
df_treated = pd.read_csv(
    treatment_url,
    sep='  ',
    header=None,
    names=column_names_dw
)

# Load the control dataset
df_control = pd.read_csv(
    control_url,
    sep='  ',
    header=None,
    names=column_names_dw
)
# Combine the treated and control datasets
df_lalonde = pd.concat([df_treated, df_control], ignore_index=True)



print(f"LaLonde (Dehejia-Wahba) dataset loaded and combined successfully. It has {df_lalonde.shape[0]} rows and {df_lalonde.shape[1]} columns.")

print("\nFirst 5 rows of the combined LaLonde dataset (Dehejia-Wahba version):")
print(df_lalonde.head().to_markdown(index=False))

print("\nColumn names and their data types for the combined dataset:")
df_lalonde.info()
print("\nNote: This dataset is constructed from separate treated and control files from Dehejia and Wahba's NBER replication data.")

/tmp/ipykernel_174/2576509615.py:13: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support regex separators (separators > 1 char and different from '\s+' are interpreted as regex); you can avoid this warning by specifying engine='python'.
  df_treated = pd.read_csv(
/tmp/ipykernel_174/2576509615.py:21: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support regex separators (separators > 1 char and different from '\s+' are interpreted as regex); you can avoid this warning by specifying engine='python'.
  df_control = pd.read_csv(


LaLonde (Dehejia-Wahba) dataset loaded and combined successfully. It has 16177 rows and 10 columns.

First 5 rows of the combined LaLonde dataset (Dehejia-Wahba version):
|   treat |   age |   education |   black |   hispan |   married |   nodegree |   re74 |   re75 |     re78 |
|--------:|------:|------------:|--------:|---------:|----------:|-----------:|-------:|-------:|---------:|
|       1 |    37 |          11 |       1 |        0 |         1 |          1 |      0 |      0 |  9930.05 |
|       1 |    22 |           9 |       0 |        1 |         0 |          1 |      0 |      0 |  3595.89 |
|       1 |    30 |          12 |       1 |        0 |         0 |          0 |      0 |      0 | 24909.5  |
|       1 |    27 |          11 |       1 |        0 |         0 |          1 |      0 |      0 |  7506.15 |
|       1 |    33 |           8 |       1 |        0 |         0 |          1 |      0 |      0 |   289.79 |

Column names and their data types for the combined dataset:
<clas

In [ ]:
print('Number of units in treatment and control groups:')
print(df_lalonde['treat'].value_counts())

print('\nDescriptive statistics for re78 when treat=1 (treatment group):')
print(df_lalonde[df_lalonde['treat'] == 1]['re78'].describe())

print('\nDescriptive statistics for re78 when treat=0 (control group):')
print(df_lalonde[df_lalonde['treat'] == 0]['re78'].describe())

Number of units in treatment and control groups:
treat
0.0    15992
1.0      185
Name: count, dtype: int64

Descriptive statistics for re78 when treat=1 (treatment group):
count      185.000000
mean      6349.143530
std       7867.402218
min          0.000000
25%        485.229800
50%       4232.309000
75%       9642.999000
max      60307.930000
Name: re78, dtype: float64

Descriptive statistics for re78 when treat=0 (control group):
count    15992.000000
mean     14846.659673
std       9647.391524
min          0.000000
25%       5669.298000
50%      16421.975000
75%      25564.670000
max      25564.670000
Name: re78, dtype: float64


In [ ]:
tau_hat=df_lalonde[df_lalonde['treat'] == 1]['re78'].mean()-df_lalonde[df_lalonde['treat'] == 0]['re78'].mean()
print(tau_hat)

-8497.516142636992


# Matching for ATT

### Select Covariate

In [ ]:
covariate_columns = [
    'age', 'education', 'black', 'hispan', 'married', 'nodegree', 're74', 're75'
]

X_treated = df_lalonde[df_lalonde['treat'] == 1][covariate_columns]
X_control = df_lalonde[df_lalonde['treat'] == 0][covariate_columns]

print("First 5 rows of X_treated (Covariates for Treated Group):")
print(X_treated.head())

print("\nFirst 5 rows of X_control (Covariates for Control Group):")
print(X_control.head())

First 5 rows of X_treated (Covariates for Treated Group):
    age  education  black  hispan  married  nodegree  re74  re75
0  37.0       11.0    1.0     0.0      1.0       1.0   0.0   0.0
1  22.0        9.0    0.0     1.0      0.0       1.0   0.0   0.0
2  30.0       12.0    1.0     0.0      0.0       0.0   0.0   0.0
3  27.0       11.0    1.0     0.0      0.0       1.0   0.0   0.0
4  33.0        8.0    1.0     0.0      0.0       1.0   0.0   0.0

First 5 rows of X_control (Covariates for Control Group):
      age  education  black  hispan  married  nodegree       re74       re75
185  45.0       11.0    0.0     0.0      1.0       1.0  21516.670  25243.550
186  21.0       14.0    0.0     0.0      0.0       0.0   3175.971   5852.565
187  38.0       12.0    0.0     0.0      1.0       0.0  23039.020  25130.760
188  48.0        6.0    0.0     0.0      1.0       1.0  24994.370  25243.550
189  18.0        8.0    0.0     0.0      1.0       1.0   1669.295  10727.610


### Implement Distance Metrics




In [ ]:
from sklearn.linear_model import LogisticRegression
from scipy.spatial import distance
import numpy as np

# 2. Define Euclidean distance function
def euclidean_distance(x1, x2):
    """Calculates the Euclidean distance between two vectors."""
    return distance.euclidean(x1, x2)

# 3. Define Mahalanobis distance function
def mahalanobis_distance(x1, x2, cov_inv):
    """Calculates the Mahalanobis distance between two vectors given the inverse covariance matrix."""
    return distance.mahalanobis(x1, x2, cov_inv)

# 4. Combine X_treated and X_control and create y_combined
X_combined = pd.concat([X_treated, X_control], ignore_index=True)
y_combined = df_lalonde['treat']

# 5. Initialize LogisticRegression model
# 6. Fit the logistic regression model
propensity_model = LogisticRegression(solver='liblinear', random_state=42)
propensity_model.fit(X_combined, y_combined)

# 7. Calculate propensity scores for all observations
df_lalonde_propensity = df_lalonde.copy()
df_lalonde_propensity['propensity_score'] = propensity_model.predict_proba(df_lalonde[covariate_columns])[:, 1]

# 8. Define propensity score distance function
def propensity_score_distance(ps1, ps2):
    """Calculates the absolute difference between two propensity scores."""
    return abs(ps1 - ps2)




### Implement Matching Algorithm



In [ ]:
from scipy.linalg import inv

def nearest_neighbor_matching(X_treated, X_control, distance_metric, cov_inv=None):
    """Performs nearest neighbor matching with replacement.

    Args:
        X_treated (pd.DataFrame): Covariates for the treated group.
        X_control (pd.DataFrame): Covariates for the control group.
        distance_metric (function): The distance function to use (e.g., euclidean_distance, mahalanobis_distance, propensity_score_distance).
        cov_inv (np.array, optional): Inverse covariance matrix for Mahalanobis distance. Defaults to None.

    Returns:
        list: A list of tuples, where each tuple contains the index of a treated unit and its matched control unit's index.
    """
    matches = []
    # Convert DataFrames to numpy arrays for efficient distance calculation
    X_treated_np = X_treated.values
    X_control_np = X_control.values

    # Extract propensity scores if distance_metric is propensity_score_distance
    if distance_metric == propensity_score_distance:
        ps_treated = X_treated['propensity_score'].values
        ps_control = X_control['propensity_score'].values

    # Keep track of the original indices of control units that have been matched
    used_control_original_indices = set()

    for i, treated_unit_np in enumerate(X_treated_np):
        min_distance = float('inf')
        best_match_control_array_idx = -1  # Index within X_control_np
        best_match_control_original_idx = -1 # Original index from X_control DataFrame

        for j, control_unit_np in enumerate(X_control_np):
            current_control_original_idx = X_control.index[j] # Get the original index

            # Include the following if matching without replacement.
            #if current_control_original_idx in used_control_original_indices:
            #    continue

            # Determine the inputs for the distance metric
            if distance_metric == propensity_score_distance:
                # For propensity score matching, the inputs are the propensity scores
                dist_input_treated = ps_treated[i]
                dist_input_control = ps_control[j]
            else:
                # For Euclidean/Mahalanobis, the inputs are the covariate vectors
                dist_input_treated = treated_unit_np
                dist_input_control = control_unit_np

            # Calculate distance based on the chosen metric
            if distance_metric == euclidean_distance:
                dist = distance_metric(dist_input_treated, dist_input_control)
            elif distance_metric == mahalanobis_distance:
                # Ensure cov_inv is provided for Mahalanobis distance
                if cov_inv is None:
                    raise ValueError("cov_inv must be provided for Mahalanobis distance.")
                dist = distance_metric(dist_input_treated, dist_input_control, cov_inv)
            elif distance_metric == propensity_score_distance:
                dist = distance_metric(dist_input_treated, dist_input_control)
            else:
                raise ValueError("Unsupported distance metric.")

            if dist < min_distance:
                min_distance = dist
                best_match_control_array_idx = j
                best_match_control_original_idx = current_control_original_idx

        # If a valid match was found for the current treated unit
        if best_match_control_array_idx != -1:
            matches.append((X_treated.index[i], best_match_control_original_idx))
            used_control_original_indices.add(best_match_control_original_idx)

    return matches

# Calculate the inverse covariance matrix for Mahalanobis distance
# using only the control group's covariates
# Need to ensure X_control for Mahalanobis does not include 'propensity_score' if it was added
control_covariates = X_control[covariate_columns]
treatment_covariates = X_treated[covariate_columns]
n0 = len(control_covariates)
n1 = len(treatment_covariates)

# Calculate the covariance matrix
cov_c = np.cov(control_covariates.transpose())
cov_t = np.cov(treatment_covariates.transpose())


cov_inv_mahalanobis = inv(cov_c * (n0)/(n0+n1) + cov_t * (n1)/(n0+n1))


In [ ]:
import numpy as np

# 1. Prepare covariate DataFrames for propensity score matching
X_treated_ps = df_lalonde_propensity[df_lalonde_propensity['treat'] == 1][['propensity_score']]
X_control_ps = df_lalonde_propensity[df_lalonde_propensity['treat'] == 0][['propensity_score']]

# 2. Perform nearest neighbor matching using Euclidean distance
matches_euclidean = nearest_neighbor_matching(
    X_treated,
    X_control,
    euclidean_distance
)
print(f"\nEuclidean matching completed. Found {len(matches_euclidean)} matches.")

# 3. Perform nearest neighbor matching using Mahalanobis distance
matches_mahalanobis = nearest_neighbor_matching(
    X_treated,
    X_control,
    mahalanobis_distance,
    cov_inv=cov_inv_mahalanobis
)
print(f"Mahalanobis matching completed. Found {len(matches_mahalanobis)} matches.")

# 4. Perform nearest neighbor matching using Propensity Score distance
# Note: The nearest_neighbor_matching function expects the 'propensity_score' column to be within X_treated_ps/X_control_ps
matches_propensity_score = nearest_neighbor_matching(
    X_treated_ps,
    X_control_ps,
    propensity_score_distance
)
print(f"Propensity Score matching completed. Found {len(matches_propensity_score)} matches.")




Euclidean matching completed. Found 185 matches.
Mahalanobis matching completed. Found 185 matches.
Propensity Score matching completed. Found 185 matches.


### Assess covariate balance using SMD


In [ ]:
# 5. Define a function to calculate Standardized Mean Difference (SMD)
def calculate_smd(treated_series, control_series):
    """Calculates the Standardized Mean Difference (SMD) between two series."""
    mean_treated = np.mean(treated_series)
    mean_control = np.mean(control_series)
    std_treated = np.std(treated_series)
    std_control = np.std(control_series)

    # Pooled standard deviation
    pooled_std = np.sqrt((std_treated**2 + std_control**2) / 2)

    if pooled_std == 0: # Avoid division by zero if std is 0
        return 0.0
    return (mean_treated - mean_control) / pooled_std

# 6. Create a function to assess balance
def assess_balance(X_treated_original, matches, df_full, covariate_columns):
    """Assess covariate balance using SMD for matched groups."""
    # Get matched control indices
    matched_control_indices = [match[1] for match in matches]

    # Extract covariates for matched treated units
    matched_treated_original = df_full.loc[[match[0] for match in matches]][covariate_columns]

    # Extract covariates for matched control units
    matched_control_covariates = df_full.loc[matched_control_indices][covariate_columns]

    smd_results = {}
    for col in covariate_columns:
        smd = calculate_smd(matched_treated_original[col], matched_control_covariates[col])
        smd_results[col] = smd
    return pd.Series(smd_results)

# 7. For each set of matches, assess and print SMDs
print("\n--- Covariate Balance Assessment ---")

# Euclidean Matching Balance
smd_euclidean = assess_balance(X_treated, matches_euclidean, df_lalonde, covariate_columns)
print("\nStandardized Mean Differences (SMD) for Euclidean Matching:")
print(smd_euclidean.to_markdown(numalign="left", stralign="left"))

# Mahalanobis Matching Balance
smd_mahalanobis = assess_balance(X_treated, matches_mahalanobis, df_lalonde, covariate_columns)
print("\nStandardized Mean Differences (SMD) for Mahalanobis Matching:")
print(smd_mahalanobis.to_markdown(numalign="left", stralign="left"))

# Propensity Score Matching Balance
smd_propensity_score = assess_balance(X_treated_ps, matches_propensity_score, df_lalonde_propensity, covariate_columns + ['propensity_score'])
print("\nStandardized Mean Differences (SMD) for Propensity Score Matching:")
print(smd_propensity_score.to_markdown(numalign="left", stralign="left"))


--- Covariate Balance Assessment ---

Standardized Mean Differences (SMD) for Euclidean Matching:
|           | 0           |
|:----------|:------------|
| age       | -0.0240141  |
| education | -0.232808   |
| black     | 1.07014     |
| hispan    | -0.0223887  |
| married   | -0.313112   |
| nodegree  | 0.228755    |
| re74      | 0.0107234   |
| re75      | 0.000447301 |

Standardized Mean Differences (SMD) for Mahalanobis Matching:
|           | 0          |
|:----------|:-----------|
| age       | 0.0114396  |
| education | 0          |
| black     | 0          |
| hispan    | 0          |
| married   | 0          |
| nodegree  | 0          |
| re74      | -0.0285741 |
| re75      | -0.034015  |

Standardized Mean Differences (SMD) for Propensity Score Matching:
|                  | 0          |
|:-----------------|:-----------|
| age              | 0.035828   |
| education        | 0.16362    |
| black            | 1.4497     |
| hispan           | -0.0439057 |
| married       

## Calculate ATT

In [ ]:
def calculate_att(df_full, matches, outcome_variable='re78'):
    """Calculates the Average Treatment Effect on the Treated (ATT) and its standard error."""
    att_values = []
    for treated_idx, control_idx in matches:
        treated_outcome = df_full.loc[treated_idx, outcome_variable]
        control_outcome = df_full.loc[control_idx, outcome_variable]
        att_values.append(treated_outcome - control_outcome)
    return np.mean(att_values), np.std(att_values)/np.sqrt(len(att_values))

# Calculate ATT for Euclidean matching
att_euclidean, se_euclidean = calculate_att(df_lalonde, matches_euclidean)
print(f"\nATT for Euclidean Matching: {att_euclidean:.2f} (SE: {se_euclidean:.2f})")

# Calculate ATT for Mahalanobis matching
att_mahalanobis, se_mahalanobis = calculate_att(df_lalonde, matches_mahalanobis)
print(f"ATT for Mahalanobis Matching: {att_mahalanobis:.2f} (SE: {se_mahalanobis:.2f})")

# Calculate ATT for Propensity Score matching
att_propensity_score, se_propensity_score = calculate_att(df_lalonde, matches_propensity_score)
print(f"ATT for Propensity Score Matching: {att_propensity_score:.2f} (SE: {se_propensity_score:.2f})")

print("\n--- Comparison of ATT Values and Standard Errors ---")
print(f"Euclidean ATT:          {att_euclidean:.2f} (SE: {se_euclidean:.2f})")
print(f"Mahalanobis ATT:        {att_mahalanobis:.2f} (SE: {se_mahalanobis:.2f})")
print(f"Propensity Score ATT:   {att_propensity_score:.2f} (SE: {se_propensity_score:.2f})")


ATT for Euclidean Matching: 1481.02 (SE: 732.10)
ATT for Mahalanobis Matching: 2088.65 (SE: 719.72)
ATT for Propensity Score Matching: 2166.68 (SE: 683.11)

--- Comparison of ATT Values and Standard Errors ---
Euclidean ATT:          1481.02 (SE: 732.10)
Mahalanobis ATT:        2088.65 (SE: 719.72)
Propensity Score ATT:   2166.68 (SE: 683.11)
